# Tracing the AL loop

This notebook walks through the **real, unmodified** orchestration code in `al-molecular` — `run_experiment.py`'s `MVEExplorer`, `molpal/models/__init__.py`'s `mve()` factory, `molpal/models/mvemodels.py`, and `surrogates.py` — using a tiny toy dataset (20 SMILES, synthetic scores) instead of the real 2.1M / 99.5M-molecule pools.

**Scope**: the 4 embedding backbones (GROVER / MoLFormer / UniMol / UniMol2, in the `compute_*_embeddings_chunk.py` scripts) are treated as a black box here — we stand in for their output with random vectors. Everything *downstream* of "here's an embedding matrix" is the real production code, run at toy scale so every step is inspectable.

Run this with the repo's `py310` conda environment as the kernel.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("/N/slate/mengjing/repos/al-molecular")
assert REPO_ROOT.exists()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
np.set_printoptions(precision=3, suppress=True)

## Step 1 — Toy data: SMILES + oracle scores

In the real pipeline this is `run_experiment.py`'s `load_library_smiles()` (pool SMILES) and `load_oracle()` (`{smiles: docking_score}`, lower = better). Here we hand-write 20 simple molecules and *synthetic* scores (not real docking) just to exercise the loop mechanics.

In [2]:
smiles = [
    "CCO",                                    # ethanol
    "c1ccccc1",                               # benzene
    "CC(=O)O",                                # acetic acid
    "CC(=O)Oc1ccccc1C(=O)O",                  # aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",           # caffeine
    "Oc1ccccc1",                              # phenol
    "CCN(CC)CC",                              # triethylamine
    "C1CCCCC1",                               # cyclohexane
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",             # ibuprofen
    "CC(=O)Nc1ccc(O)cc1",                     # acetaminophen
    "c1ccc2ccccc2c1",                         # naphthalene
    "CCOC(=O)c1ccccc1",                       # ethyl benzoate
    "NC(=O)c1ccccc1",                         # benzamide
    "Clc1ccccc1",                             # chlorobenzene
    "Cc1ccccc1",                              # toluene
    "OCC(O)CO",                               # glycerol
    "CC(C)O",                                 # isopropanol
    "c1ccncc1",                               # pyridine
    "NCCO",                                   # ethanolamine
    "CC(=O)C",                                # acetone
]

# Synthetic "docking scores" (kcal/mol, lower = better) -- NOT real chemistry,
# just deterministic numbers so the loop has something to rank and acquire.
rng = np.random.RandomState(0)
scores = rng.uniform(-10.0, -5.0, size=len(smiles))
oracle = dict(zip(smiles, scores))

print("best 5 (lowest = best):")
for s, v in sorted(oracle.items(), key=lambda kv: kv[1])[:5]:
    print(f"  {v:6.2f}  {s}")
print(f"\n{len(oracle)} molecules total, best={min(oracle.values()):.2f}, worst={max(oracle.values()):.2f}")

best 5 (lowest = best):
   -9.90  CC(C)O
   -9.64  Cc1ccccc1
   -9.56  OCC(O)CO
   -8.08  CC(=O)Nc1ccc(O)cc1
   -7.88  CN1C=NC2=C1C(=O)N(C(=O)N2C)C

20 molecules total, best=-9.90, worst=-5.18


## Step 2 — Fake embeddings (standing in for a backbone)

In the real pipeline, `compute_molformer_embeddings_chunk.py` (etc.) turns each SMILES into a fixed-length vector — 768-d for MoLFormer, 512-d for UniMol, 1600-d for GROVER — written to disk, then `molpal/featurizer.py`'s `EmbeddingFeaturizer` loads them back as `emb_dict = {backbone_name: (N, D) array}` aligned to `pool_smiles` order. **That's the piece we're skipping.** Here `emb_dict` is one made-up 8-d "backbone" per molecule, random but fixed by seed.

In [3]:
EMB_DIM = 8  # tiny, arbitrary -- real backbones use 512 / 768 / 1600

emb_rng = np.random.RandomState(1)
toy_embeddings = emb_rng.randn(len(smiles), EMB_DIM).astype(np.float32)

emb_dict = {"toybackbone": toy_embeddings}
print(emb_dict["toybackbone"].shape)

(20, 8)


## Step 3 — Build the surrogate (real factory code)

`molpal/models/__init__.py`'s `mve()` is the real factory `MVEExplorer` calls. `surrogate_type="single"` builds a `SingleBackboneMVESurrogate` (dual MVE heads, defined in `surrogates.py`) wrapped in a `SingleBackboneEmbeddingModel` (defined in `molpal/models/mvemodels.py`) that knows how to turn SMILES into embedding rows via `_get_X()`.

In [4]:
from molpal.models import mve as build_mve

dims = {k: v.shape[1] for k, v in emb_dict.items()}
print("embedding dims per backbone:", dims)

model = build_mve(
    surrogate_type="single",
    backbone="toybackbone",
    emb_dict=emb_dict,
    pool_smiles=smiles,
    dataset_name="Toy",
)

print(type(model).__name__)            # SingleBackboneEmbeddingModel
print(type(model.surrogate).__name__)  # SingleBackboneMVESurrogate
print(model.surrogate._model)          # the actual _DualMVEModel: backbone -> two MVE heads

embedding dims per backbone: {'toybackbone': 8}
SingleBackboneEmbeddingModel
SingleBackboneMVESurrogate
_DualMVEModel(
  (backbone): _LightweightBackbone(
    (net): Sequential(
      (0): Linear(in_features=8, out_features=256, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): Dropout(p=0.25, inplace=False)
      (4): Linear(in_features=256, out_features=128, bias=True)
      (5): ReLU()
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): Dropout(p=0.25, inplace=False)
    )
  )
  (head1): _MVEHead(
    (mu_net): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.25, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
    (var_net): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.25, inplace=Fa

## Step 4 — Macro view: run the *real* `MVEExplorer` end-to-end

This is the unmodified class from `run_experiment.py` — the exact code that runs the full 2.1M / 99.5M-molecule experiments — just fed our toy `emb_dict` / `smiles` / `oracle` and small round sizes. Every print statement below is production code, not notebook code.

In [5]:
import run_experiment as exp

# Disposable scratch output -- not the real runs/ used by actual experiments.
toy_run_dir = REPO_ROOT / "notebooks" / "_toy_run"

explorer = exp.MVEExplorer(
    emb_dict=emb_dict,
    pool_smiles=smiles,
    oracle=oracle,
    surrogate_type="single",
    backbone="toybackbone",
    acq="greedy",
    init_size=6,
    batch_size=4,
    n_rounds=3,
    topk=5,
    run_dir=toy_run_dir,
    seed=0,
)

history = explorer.run()
history

[init] 6 random molecules  best=-7.882 kcal/mol
  Round 01/3  labeled=10  best=-9.899 kcal/mol  top-5 recall=80.0%  (1.8s)
  Round 02/3  labeled=14  best=-9.899 kcal/mol  top-5 recall=80.0%  (0.3s)
  Round 03/3  labeled=18  best=-9.899 kcal/mol  top-5 recall=100.0%  (0.3s)

[done] results -> /N/slate/mengjing/repos/al-molecular/notebooks/_toy_run


[{'round': 1,
  'n_labeled': 10,
  'best_score': -9.89890801279837,
  'topk_recall': 0.8,
  'elapsed': 1.75},
 {'round': 2,
  'n_labeled': 14,
  'best_score': -9.89890801279837,
  'topk_recall': 0.8,
  'elapsed': 0.28},
 {'round': 3,
  'n_labeled': 18,
  'best_score': -9.89890801279837,
  'topk_recall': 1.0,
  'elapsed': 0.28}]

## Step 5 — Micro view: one round's internals, line by line

`MVEExplorer.run()` is one loop body repeated `n_rounds` times. Below is that same loop body run manually with every intermediate value exposed. We build a fresh explorer (same toy data, same seed) so the initial state matches what Step 4 started from.

In [6]:
explorer2 = exp.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=smiles, oracle=oracle,
    surrogate_type="single", backbone="toybackbone", acq="greedy",
    init_size=6, batch_size=4, n_rounds=3, topk=5,
    run_dir=toy_run_dir.parent / "_toy_run_manual", seed=0,
)

print("labeled_idx (randomly chosen init set):", sorted(explorer2.labeled_idx))
print("labeled smiles:", [explorer2.pool_smiles[i] for i in sorted(explorer2.labeled_idx)])

# ---- one round of MVEExplorer.run(), unrolled ----

# 1. Gather the labeled (training) set
idx = list(explorer2.labeled_idx)
xs = [explorer2.pool_smiles[i] for i in idx]
ys = explorer2._sign * np.array(
    [explorer2.labeled_scores[explorer2.pool_smiles[i]] for i in idx], dtype=np.float32
)
print("\n[1] training set:", len(xs), "molecules")
print("    ys (sign-flipped so higher = better):", ys)

# 2. Train the surrogate: SingleBackboneEmbeddingModel.train() -> _get_X() -> surrogate.fit()
explorer2.model.train(xs, ys)
print("\n[2] trained. surrogate y-normalisation (mean, std) from this training set:",
      explorer2.model.surrogate._ym, explorer2.model.surrogate._ys)

# 3. Build the remaining pool (everything not yet labeled)
n = len(explorer2.pool_smiles)
mask = np.ones(n, bool)
for i in explorer2.labeled_idx:
    mask[i] = False
pool_idx = np.where(mask)[0]
pool_smi = [explorer2.pool_smiles[i] for i in pool_idx]
print(f"\n[3] remaining pool: {len(pool_smi)} molecules")

# 4. Predict mean + variance over the pool
mu, var = explorer2.model.get_means_and_vars(pool_smi)
print("\n[4] predictions:")
for s, m, v in zip(pool_smi, mu, var):
    print(f"    {m:7.3f}  var={v:.3f}  {s}")

# 5. Acquisition: greedy just ranks by mu (no variance term)
scores = explorer2.acq_fn(mu)
print("\n[5] acquisition scores (greedy = mu, unchanged):", scores)

# 6. Pick the top batch_size molecules
top_local = np.argsort(scores)[::-1][: explorer2.batch_size]
selected = pool_idx[top_local]
print("\n[6] selected for labeling this round:")
for i in selected:
    print(f"    {explorer2.pool_smiles[i]}  (true oracle score={explorer2.oracle[explorer2.pool_smiles[i]]:.2f})")

[init] 6 random molecules  best=-7.882 kcal/mol
labeled_idx (randomly chosen init set): [0, 4, 5, 8, 10, 12]
labeled smiles: [np.str_('CCO'), np.str_('CN1C=NC2=C1C(=O)N(C(=O)N2C)C'), np.str_('Oc1ccccc1'), np.str_('CC(C)Cc1ccc(cc1)C(C)C(=O)O'), np.str_('c1ccc2ccccc2c1'), np.str_('NC(=O)c1ccccc1')]

[1] training set: 6 molecules
    ys (sign-flipped so higher = better): [7.256 7.882 6.771 5.182 6.041 7.16 ]

[2] trained. surrogate y-normalisation (mean, std) from this training set: 6.7151713371276855 0.8812480072839356

[3] remaining pool: 14 molecules

[4] predictions:
      7.665  var=0.391  c1ccccc1
      7.445  var=0.345  CC(=O)O
      6.626  var=0.332  CC(=O)Oc1ccccc1C(=O)O
      6.838  var=0.512  CCN(CC)CC
      7.013  var=0.368  C1CCCCC1
      6.984  var=0.324  CC(=O)Nc1ccc(O)cc1
      6.997  var=0.529  CCOC(=O)c1ccccc1
      6.164  var=0.394  Clc1ccccc1
      7.137  var=0.538  Cc1ccccc1
      7.803  var=0.290  OCC(O)CO
      7.311  var=0.484  CC(C)O
      7.058  var=0.515  c1ccnc

## Step 6 — Acquisition functions up close

`molpal/acquirer/metrics.py`'s `get_metric()` returns one of these. `greedy(mu)` just returns `mu` unchanged; `ucb(mu, var, beta=2)` returns `mu + beta * sqrt(var)`, so it rewards molecules the model is both optimistic *and* uncertain about. Same `mu` / `var` from Step 5, compared side by side.

In [7]:
from molpal.acquirer.metrics import get_metric
import pandas as pd

greedy_fn = get_metric("greedy")
ucb_fn = get_metric("ucb")

greedy_scores = greedy_fn(mu)
ucb_scores = ucb_fn(mu, var)

pd.DataFrame({
    "smiles": pool_smi,
    "mu": mu,
    "var": var,
    "greedy_score": greedy_scores,
    "ucb_score": ucb_scores,
}).sort_values("ucb_score", ascending=False)

,smiles,mu,var,greedy_score,ucb_score
0,c1ccccc1,7.665267,0.390968,7.665267,8.915815
9,OCC(O)CO,7.802962,0.289522,7.802962,8.879107
10,CC(C)O,7.310702,0.484264,7.310702,8.702484
1,CC(=O)O,7.444955,0.345361,7.444955,8.620303
8,Cc1ccccc1,7.136862,0.538400,7.136862,8.604377
11,c1ccncc1,7.057688,0.514752,7.057688,8.492613
6,CCOC(=O)c1ccccc1,6.997446,0.528902,6.997446,8.451960
3,CCN(CC)CC,6.838236,0.512199,6.838236,8.269598
4,C1CCCCC1,7.012937,0.368120,7.012937,8.226395
13,CC(=O)C,6.755404,0.472490,6.755404,8.130161


## Where this maps back to the real pipeline

| Toy notebook step | Real file |
|---|---|
| `smiles` + `oracle` | `run_experiment.py::load_library_smiles()` / `load_oracle()` |
| `emb_dict` (fake here) | `compute_{grover,molformer,unimol,unimol2}_embeddings_chunk.py` → `shared_embedding_store.py` → `stitch_embedding_chunks.py` → `molpal/featurizer.py::EmbeddingFeaturizer.load()` |
| `build_mve(...)` | `molpal/models/__init__.py::mve()` |
| `SingleBackboneEmbeddingModel` / `_get_X()` | `molpal/models/mvemodels.py` |
| `SingleBackboneMVESurrogate` (`_DualMVEModel`, MVE heads) | `surrogates.py` |
| `MVEExplorer.run()` | `run_experiment.py` (has a near-duplicate `MolPALExplorer` for the fingerprint/MPN baseline) |
| `get_metric("greedy" / "ucb")` | `molpal/acquirer/metrics.py` |

**Not touched by this notebook, worth a follow-up trace:**
- **Fusion surrogates** (`EnsembleFusionSurrogate`, `LearnedFusionSurrogate` in `surrogates.py`) — same `fit`/`predict` interface, but combine multiple backbones' embeddings/predictions. Swap `surrogate_type="single"` for `"ensemble"` / `"learned"` and pass a multi-key `emb_dict` (e.g. two fake backbones with different dims) to trace those.
- **Fine-tuning surrogates** (`backbone_finetuner.py`) — the only place a backbone's weights actually change during the AL loop; can't be faked the way we did here, since it retrains the pooling/projection itself.
- **The legacy `molpal/` fingerprint path** (`MolPALExplorer` + `--mode molpal --model mpn`) — structurally the same loop, different featurization (Morgan/atom-pair fingerprints instead of embeddings).